# 🔴 Solution: LoRA Linear Layer

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ✅ SOLUTION

class LoRALinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, rank: int, alpha: float = 1.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        
        self.W = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.W)

        if rank>0:
            self.A = nn.Parameter(torch.empty(rank, in_features))
            self.B = nn.Parameter(torch.zeros(out_features, rank))
            nn.init.kaiming_uniform_(self.A)
        else:
            self.A = nn.Parameter(torch.empty(0, in_features), requires_grad=False)
            self.B = nn.Parameter(torch.empty(out_features, 0), requires_grad=False)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.linear(x, self.W)

        if self.rank > 0:
            lora_out = F.linear(x, self.A)
            lora_out = F.linear(lora_out, self.B)
            out = out + self.alpha/self.rank * lora_out

        return out

In [ ]:
from torch_judge import check
check("lora")